### Gold — Tables de faits

Construit `fact_ventes` et `fact_retours` en remplaçant les attributs texte (region, produit...) par les clés de substitution des dimensions déjà créées dans `gold_dimensions.py`. 

**Dépendance : ce notebook suppose que gold_dimensions.py a déjà tourné.**

In [0]:
from pyspark.sql import functions as F

In [0]:
df_ventes = spark.table("silver.ventes.ventes_clean")
df_retours = spark.table("silver.ventes.retours_clean")

dim_produit = spark.table("gold.ventes.dim_produit")
dim_region = spark.table("gold.ventes.dim_region")
dim_client = spark.table("gold.ventes.dim_client")
dim_vendeur = spark.table("gold.ventes.dim_vendeur")
dim_temps = spark.table("gold.ventes.dim_temps")

In [0]:
# MAGIC Chaque jointure remplace un attribut texte par sa clé de substitution.
# MAGIC On joint sur la clé naturelle (`id_produit`, `region`...) pour retrouver
# MAGIC la clé technique correspondante, puis on ne garde que cette dernière —
# MAGIC c'est le principe même du modèle en étoile : le fait ne stocke que des
# MAGIC clés courtes, jamais les libellés complets.

fact_ventes = (
    df_ventes
    .join(dim_produit.select("id_produit", "sk_produit"), on="id_produit", how="left")
    .join(dim_region.select("region", "sk_region"), on="region", how="left")
    .join(dim_client.select("id_client", "sk_client"), on="id_client", how="left")
    .join(dim_vendeur.select("id_vendeur", "sk_vendeur"), on="id_vendeur", how="left")
    .join(
        dim_temps.select(F.col("date_vente"), F.col("sk_temps")),
        on="date_vente",
        how="left"
    )
    .select(
        "id_vente",
        "sk_temps",
        "sk_produit",
        "sk_region",
        "sk_client",
        "sk_vendeur",
        "canal_vente",   # dimension dégénérée, gardée directement ici
        "quantite",
        "prix_unitaire",
        "montant_total",
    )
)

display(fact_ventes.limit(10))

In [0]:
# MAGIC Une jointure `left` qui ne trouve pas de correspondance produit un `NULL`
# MAGIC silencieusement plutôt qu'une erreur — c'est le piège classique des
# MAGIC jointures de dimensions. On vérifie explicitement qu'aucune clé technique
# MAGIC n'est nulle avant d'écrire la table : un NULL ici signifierait qu'une
# MAGIC valeur existe dans les faits mais pas dans la dimension correspondante,
# MAGIC ce qui ne devrait jamais arriver puisque les dimensions sont construites
# MAGIC à partir des mêmes données silver.


cles_a_verifier = ["sk_temps", "sk_produit", "sk_region", "sk_client", "sk_vendeur"]

for cle in cles_a_verifier:
    n_nulls = fact_ventes.filter(F.col(cle).isNull()).count()
    statut = "OK" if n_nulls == 0 else "ATTENTION"
    print(f"[{statut}] {cle} : {n_nulls} valeurs nulles")

In [0]:
# MAGIC Plus simple : on rattache uniquement `sk_temps` (date du retour). Le lien
# MAGIC vers le produit ou le client d'une vente retournée se fait via une
# MAGIC jointure entre fact_retours et fact_ventes au moment de l'analyse
# MAGIC (`id_vente` reste la clé de liaison entre les deux tables de faits),
# MAGIC pas en dupliquant ces attributs ici.

fact_retours = (
    df_retours
    .join(
        dim_temps.select(F.col("date_vente").alias("date_retour"), F.col("sk_temps")),
        on="date_retour",
        how="left"
    )
    .select(
        "id_retour",
        "id_vente",
        "sk_temps",
        "motif_retour",
        "statut",
        "montant_rembourse",
    )
)

display(fact_retours.limit(10))

In [0]:
n_nulls_temps = fact_retours.filter(F.col("sk_temps").isNull()).count()
statut = "OK" if n_nulls_temps == 0 else "ATTENTION"
print(f"[{statut}] sk_temps : {n_nulls_temps} valeurs nulles")

In [0]:
(
    fact_ventes.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.ventes.fact_ventes")
)

(
    fact_retours.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.ventes.fact_retours")
)

print(f"gold.ventes.fact_ventes : {fact_ventes.count()} lignes")
print(f"gold.ventes.fact_retours : {fact_retours.count()} lignes")

In [0]:
%sql
SELECT
r.region, p.categorie, SUM(f.montant_total) AS ca_total
FROM gold.ventes.fact_ventes f
JOIN gold.ventes.dim_region r ON f.sk_region = r.sk_region
JOIN gold.ventes.dim_produit p ON f.sk_produit = p.sk_produit
GROUP BY r.region, p.categorie
ORDER BY ca_total DESC
LIMIT 10;